# Julia is fast

Very often, benchmarks are used to compare languages.  These benchmarks can lead to long discussions, first as to exactly what is being benchmarked and secondly what explains the differences.  These simple questions can sometimes get more complicated than you at first might imagine.

The purpose of this notebook is for you to see a simple benchmark for yourself.  One can read the notebook and see what happened on the author's Macbook Pro with a 4-core Intel Core I7, or run the notebook yourself.

(This material began life as a wonderful lecture by Steven Johnson at MIT: https://github.com/stevengj/18S096/blob/master/lectures/lecture1/Boxes-and-registers.ipynb.)

# Outline of this notebook

- Define the sum function
- Implementations & benchmarking of sum in...
    - C (hand-written)
    - C (hand-written with -ffast-math)
    - python (built-in)
    - python (numpy)
    - python (hand-written)
    - Julia (built-in)
    - Julia (hand-written)
    - Julia (hand-written with SIMD)
- Summary of benchmarks

# `sum`: An easy enough function to understand

Consider the  **sum** function `sum(a)`, which computes
$$
\mathrm{sum}(a) = \sum_{i=1}^n a_i,
$$
where $n$ is the length of `a`.

In [1]:
a = rand(10^7) # 1D vector of random numbers, uniform on [0,1)

10000000-element Vector{Float64}:
 0.4425718651403565
 0.6449802982799411
 0.3392430910065808
 0.23384609087823593
 0.6484198770526456
 0.037515602989179
 0.25228250901460614
 0.14993521886150607
 0.7234944078730431
 0.09426518102898773
 ⋮
 0.9997022271156051
 0.06938426665277941
 0.5879682776721333
 0.5181652046232916
 0.11367869693402677
 0.3188222010911661
 0.1760395113648633
 0.6670037881494489
 0.8469141911563814

In [2]:
sum(a)

5.001869133866429e6

The expected result is 0.5 * 10^7, since the mean of each entry is 0.5

# Benchmarking a few ways in a few languages

In [3]:
@time sum(a)

  0.000983 seconds (1 allocation: 16 bytes)


5.001869133866429e6

In [4]:
@time sum(a)

  0.000934 seconds (1 allocation: 16 bytes)


5.001869133866429e6

In [5]:
@time sum(a)

  0.000898 seconds (1 allocation: 16 bytes)


5.001869133866429e6

The `@time` macro can yield noisy results, so it's not our best choice for benchmarking!

Luckily, Julia has a `BenchmarkTools.jl` package to make benchmarking easy and accurate:

In [50]:
using Pkg
Pkg.add("BenchmarkTools")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Compat ───────── v4.18.1
   Installed BenchmarkTools ─ v1.6.3
    Updating `~/.julia/environments/v1.12/Project.toml`
  [6e4b80f9] + BenchmarkTools v1.6.3
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.6.3
  [34da2185] + Compat v4.18.1
  [9abbd945] + Profile v1.11.0
Precompiling packages...
    355.5 ms  ✓ Compat
    454.7 ms  ✓ Compat → CompatLinearAlgebraExt
    743.1 ms  ✓ BenchmarkTools
  3 dependencies successfully precompiled in 2 seconds. 193 already precompiled.


In [51]:
using BenchmarkTools  

#  1. The C language

C is often considered the gold standard: difficult on the human, nice for the machine. Getting within a factor of 2 of C is often satisfying. Nonetheless, even within C, there are many kinds of optimizations possible that a naive C writer may or may not get the advantage of.

The current author does not speak C, so he does not read the cell below, but is happy to know that you can put C code in a Julia session, compile it, and run it. Note that the `"""` wrap a multi-line string.

In [53]:
using Libdl
C_code = """
#include <stddef.h>
double c_sum(size_t n, double *X) {
    double s = 0.0;
    for (size_t i = 0; i < n; ++i) {
        s += X[i];
    }
    return s;
}
"""

const Clib = tempname()   # make a temporary file


# compile to a shared library by piping C_code to gcc
# (works only if you have gcc installed):

open(`gcc -fPIC -O3 -msse3 -xc -shared -o $(Clib * "." * Libdl.dlext) -`, "w") do f
    print(f, C_code) 
end

# define a Julia function that calls the C function:
c_sum(X::Array{Float64}) = ccall(("c_sum", Clib), Float64, (Csize_t, Ptr{Float64}), length(X), X)

clang: error: unsupported option '-msse3' for target 'arm64-apple-darwin25.0.0'


ProcessFailedException: failed process: Process(`gcc -fPIC -O3 -msse3 -xc -shared -o /var/folders/2_/kz_ldg212f16_7lfh1b497qm0000gn/T/jl_5U36bwbWnA.dylib -`, ProcessExited(1)) [1]


In [54]:
c_sum(a)

UndefVarError: UndefVarError: `c_sum` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [10]:
c_sum(a) ≈ sum(a) # type \approx and then <TAB> to get the ≈ symbolb

UndefVarError: UndefVarError: `c_sum` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
c_sum(a) - sum(a)  

UndefVarError: UndefVarError: `c_sum` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
≈  # alias for the `isapprox` function

isapprox (generic function with 13 methods)

In [13]:
?isapprox

Base.Meta.ParseError: ParseError:
# Error @ /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X26sZmlsZQ==.jl:1:1
?isapprox
╙ ── not a unary operator

We can now benchmark the C code directly from Julia:

In [14]:
c_bench = @benchmark c_sum($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X31sZmlsZQ==.jl:1

In [15]:
println("C: Fastest time was $(minimum(c_bench.times) / 1e6) msec")

UndefVarError: UndefVarError: `c_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
d = Dict()  # a "dictionary", i.e. an associative array
d["C"] = minimum(c_bench.times) / 1e6  # in milliseconds
d

UndefVarError: UndefVarError: `c_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
using Plots
gr()

Plots.GRBackend()

In [18]:
using Statistics # bring in statistical support for standard deviations
t = c_bench.times / 1e6 # times in milliseconds
m, σ = minimum(t), std(t)

histogram(t, bins=500,
    xlim=(m - 0.01, m + σ),
    xlabel="milliseconds", ylabel="count", label="")

UndefVarError: UndefVarError: `c_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 2. C with -ffast-math

If we allow C to re-arrange the floating point operations, then it'll vectorize with SIMD (single instruction, multiple data) instructions.

In [19]:
const Clib_fastmath = tempname()   # make a temporary file

# The same as above but with a -ffast-math flag added
open(`gcc -fPIC -O3 -msse3 -xc -shared -ffast-math -o $(Clib_fastmath * "." * Libdl.dlext) -`, "w") do f
    print(f, C_code) 
end

# define a Julia function that calls the C function:
c_sum_fastmath(X::Array{Float64}) = ccall(("c_sum", Clib_fastmath), Float64, (Csize_t, Ptr{Float64}), length(X), X)

clang: error: unsupported option '-msse3' for target 'arm64-apple-darwin25.0.0'


ProcessFailedException: failed process: Process(`gcc -fPIC -O3 -msse3 -xc -shared -ffast-math -o /var/folders/2_/kz_ldg212f16_7lfh1b497qm0000gn/T/jl_bOVq3ClWbU.dylib -`, ProcessExited(1)) [1]


In [20]:
c_fastmath_bench = @benchmark $c_sum_fastmath($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X41sZmlsZQ==.jl:1

In [21]:
d["C -ffast-math"] = minimum(c_fastmath_bench.times) / 1e6  # in milliseconds

UndefVarError: UndefVarError: `c_fastmath_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 3. Python's built in `sum` 

The `PyCall` package provides a Julia interface to Python:

In [22]:
# using Pkg; Pkg.add("PyCall")
using PyCall

ArgumentError: ArgumentError: Package PyCall not found in current path.
- Run `import Pkg; Pkg.add("PyCall")` to install the PyCall package.

In [23]:
# get the Python built-in "sum" function:
pysum = pybuiltin("sum")

UndefVarError: UndefVarError: `pybuiltin` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [24]:
pysum(a)

UndefVarError: UndefVarError: `pysum` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [25]:
pysum(a) ≈ sum(a)

UndefVarError: UndefVarError: `pysum` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [26]:
py_list_bench = @benchmark $pysum($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X52sZmlsZQ==.jl:1

In [27]:
d["Python built-in"] = minimum(py_list_bench.times) / 1e6
d

UndefVarError: UndefVarError: `py_list_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 4. Python: `numpy` 

## Takes advantage of hardware "SIMD", but only works when it works.

`numpy` is an optimized C library, callable from Python.
It may be installed within Julia as follows:

In [28]:
# using Pkg; Pkg.add("Conda")
using Conda

ArgumentError: ArgumentError: Package Conda not found in current path.
- Run `import Pkg; Pkg.add("Conda")` to install the Conda package.

In [29]:
# Conda.add("numpy")

In [30]:
numpy_sum = pyimport("numpy")["sum"]

py_numpy_bench = @benchmark $numpy_sum($a)

UndefVarError: UndefVarError: `pyimport` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [31]:
numpy_sum(a)

UndefVarError: UndefVarError: `numpy_sum` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [32]:
numpy_sum(a) ≈ sum(a)

UndefVarError: UndefVarError: `numpy_sum` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [33]:
d["Python numpy"] = minimum(py_numpy_bench.times) / 1e6
d

UndefVarError: UndefVarError: `py_numpy_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 5. Python, hand-written 

In [34]:
py"""
def py_sum(A):
    s = 0.0
    for a in A:
        s += a
    return s
"""

sum_py = py"py_sum"

LoadError: LoadError: UndefVarError: `@py_str` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X65sZmlsZQ==.jl:1

In [35]:
py_hand = @benchmark $sum_py($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X66sZmlsZQ==.jl:1

In [36]:
sum_py(a)

UndefVarError: UndefVarError: `sum_py` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [37]:
sum_py(a) ≈ sum(a)

UndefVarError: UndefVarError: `sum_py` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [38]:
d["Python hand-written"] = minimum(py_hand.times) / 1e6
d

UndefVarError: UndefVarError: `py_hand` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 6. Julia (built-in) 

## Written directly in Julia, not in C!

In [39]:
@which sum(a)

sum(a::AbstractArray; dims, kw...)
     @ Base reducedim.jl:979

In [40]:
j_bench = @benchmark sum($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y105sZmlsZQ==.jl:1

In [41]:
d["Julia built-in"] = minimum(j_bench.times) / 1e6
d

UndefVarError: UndefVarError: `j_bench` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 7. Julia (hand-written) 

In [42]:
function mysum(A)   
    s = 0.0 # s = zero(eltype(a))
    for a in A
        s += a
    end
    s
end

mysum (generic function with 1 method)

In [43]:
j_bench_hand = @benchmark mysum($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y112sZmlsZQ==.jl:1

In [44]:
d["Julia hand-written"] = minimum(j_bench_hand.times) / 1e6
d

UndefVarError: UndefVarError: `j_bench_hand` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 8. Julia (hand-written w. simd) 

In [45]:
function mysum_simd(A)   
    s = 0.0 # s = zero(eltype(A))
    @simd for a in A
        s += a
    end
    s
end

mysum_simd (generic function with 1 method)

In [46]:
j_bench_hand_simd = @benchmark mysum_simd($a)

LoadError: LoadError: UndefVarError: `@benchmark` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/sheikhnaumanibrahimahmed/Desktop/GitRepos/cst_longest_maximal_chains/DAGs/Julia version/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y116sZmlsZQ==.jl:1

In [47]:
mysum_simd(a)

5.001869133866412e6

In [48]:
d["Julia hand-written simd"] = minimum(j_bench_hand_simd.times) / 1e6
d

UndefVarError: UndefVarError: `j_bench_hand_simd` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Summary

In [49]:
for (key, value) in sort(collect(d), by=last)
    println(rpad(key, 25, "."), lpad(round(value; digits=1), 6, "."))
end